# 06 — Multimodal Fusion

Combines all three encoders: SELFIES Transformer (1D) + GINEConv (2D) + SchNet with
Boltzmann-weighted conformer aggregation (3D, using the `weighted_mean` variant locked
in from notebook 05).

Trains two variants for the required ablation:
- **concat** — simple concatenation baseline
- **gated** — learned per-molecule modality trust weights

These are the final two rows of the ablation table, alongside the 1D/2D/3D-only numbers
already collected.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from pathlib import Path

import yaml
import torch
from torch.utils.data import DataLoader

from src.utils import load_json, save_json, set_seed
from src.featurizers import smiles_to_selfies, PAD
from src.datasets import MultimodalDataset, multimodal_collate
from src.models import MultimodalModel
from src.train_utils import train_binary_classifier, evaluate

with open("../config/model_fusion.yaml") as f:
    cfg = yaml.safe_load(f)

set_seed(cfg["train"]["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## Load split + the train-only vocab from notebook 03

Reusing the same vocab (not rebuilding) keeps the SELFIES tokenization identical to the 1D-only run — important for a fair comparison.

In [2]:
splits = load_json(cfg["paths"]["splits_file"])
vocab = load_json(cfg["paths"]["vocab_file"])
print(f"Vocab size: {len(vocab)}")

selfies_by_split = {
    split_name: [smiles_to_selfies(s) for s in splits[split_name]["smiles"]]
    for split_name in ["train", "valid", "test"]
}


Vocab size: 63


In [3]:
mm_datasets = {
    split_name: MultimodalDataset(
        ids=splits[split_name]["id"],
        smiles_list=splits[split_name]["smiles"],
        selfies_list=selfies_by_split[split_name],
        labels=splits[split_name]["y"],
        vocab=vocab,
        max_len=cfg["encoders"]["selfies"]["max_len"],
        conformers_dir=cfg["paths"]["conformers_dir"],
    )
    for split_name in ["train", "valid", "test"]
}
print({k: len(v) for k, v in mm_datasets.items()})

sample_loader = DataLoader(mm_datasets["train"], batch_size=4, collate_fn=multimodal_collate)
sample_batch = next(iter(sample_loader))
{k: (v.shape if hasattr(v, 'shape') else v) for k, v in sample_batch.items() if k != 'graph'}


[12:09:53] WARNING: not removing hydrogen atom without neighbors
[12:09:55] WARNING: not removing hydrogen atom without neighbors


{'train': 453, 'valid': 66, 'test': 132}


{'input_ids': torch.Size([4, 128]),
 'conf_atom_z': torch.Size([1171]),
 'conf_atom_pos': torch.Size([1171, 3]),
 'conf_atom_conf_batch': torch.Size([1171]),
 'conf_conf_mol_batch': torch.Size([21]),
 'conf_conf_weights': torch.Size([21]),
 'conf_num_mols': 4,
 'label': torch.Size([4])}

## Shared training helper for both fusion variants

In [4]:
from src.graph_featurizer import ATOM_FEAT_DIM, BOND_FEAT_DIM

def train_fusion_variant(fusion_type, experiment_dir):
    loaders = {
        split_name: DataLoader(
            mm_datasets[split_name], batch_size=cfg["train"]["batch_size"],
            shuffle=(split_name == "train"), collate_fn=multimodal_collate,
        )
        for split_name in ["train", "valid", "test"]
    }

    model = MultimodalModel(
        vocab_size=len(vocab), pad_id=vocab[PAD],
        selfies_kwargs=dict(cfg["encoders"]["selfies"]),
        graph_kwargs={"atom_feat_dim": ATOM_FEAT_DIM, "bond_feat_dim": BOND_FEAT_DIM, **{
            k: v for k, v in cfg["encoders"]["graph"].items()
        }},
        schnet_kwargs=dict(cfg["encoders"]["schnet"]),  # dict() copy -- forward pops agg_mode internally
        fusion_type=fusion_type,
        fusion_dim=cfg["fusion"]["fusion_dim"],
    )
    print(f"[{fusion_type}] params: {sum(p.numel() for p in model.parameters()):,}")

    def forward_fn(model, batch, device):
        graph = batch["graph"].to(device)
        return model(
            batch["input_ids"].to(device),
            graph.x, graph.edge_index, graph.edge_attr, graph.batch,
            batch["conf_atom_z"].to(device), batch["conf_atom_pos"].to(device),
            batch["conf_atom_conf_batch"].to(device), batch["conf_conf_mol_batch"].to(device),
            batch["conf_conf_weights"].to(device), batch["conf_num_mols"],
        )

    model, train_info = train_binary_classifier(
        model, loaders["train"], loaders["valid"], forward_fn, cfg, experiment_dir, device
    )
    val_metrics = evaluate(model, loaders["valid"], forward_fn, device)
    print(f"[{fusion_type}] val AUROC: {val_metrics['auroc']:.4f}")

    save_json(
        {"model": f"multimodal_{fusion_type}", **val_metrics, **train_info, "config": cfg},
        Path(experiment_dir) / "metrics.json",
    )
    return model, val_metrics["auroc"]


## Concat baseline

In [5]:
_, auroc_concat = train_fusion_variant("concat", cfg["paths"]["experiment_dir_concat"])


[concat] params: 970,626


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\bld\libtorch_1780241954714\work\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


epoch   1  train_loss=0.8032  val_loss=0.5440  val_auroc=0.6713


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch   2  train_loss=0.5770  val_loss=0.5225  val_auroc=0.6787


...

epoch  19  train_loss=0.1083  val_loss=0.5535  val_auroc=0.8300


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch  20  train_loss=0.1195  val_loss=0.4924  val_auroc=0.8638


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch  21  train_loss=0.0897  val_loss=0.5930  val_auroc=0.8675
Early stopping at epoch 21 (no val AUROC improvement for 10 epochs)
[concat] val AUROC: 0.8838


## Gated fusion

In [6]:
gated_model, auroc_gated = train_fusion_variant("gated", cfg["paths"]["experiment_dir_gated"])


[gated] params: 1,078,021


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch   1  train_loss=0.6241  val_loss=0.6146  val_auroc=0.6138


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch   2  train_loss=0.5431  val_loss=0.5877  val_auroc=0.6275


...

epoch  25  train_loss=0.2379  val_loss=0.5542  val_auroc=0.7600


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch  26  train_loss=0.2125  val_loss=0.4888  val_auroc=0.8125


c:\Users\ajaya\anaconda3\envs\chemprop\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


epoch  27  train_loss=0.2743  val_loss=0.5695  val_auroc=0.7338
Early stopping at epoch 27 (no val AUROC improvement for 10 epochs)
[gated] val AUROC: 0.8462


## Full ablation table

In [7]:
import pandas as pd

d1 = load_json("../experiments/1d_only/metrics.json")
d2 = load_json("../experiments/2d_only/metrics.json")
d3_uniform = load_json("../experiments/3d_uniform_mean/metrics.json")
d3_weighted = load_json("../experiments/3d_only/metrics.json")
d3_attn = load_json("../experiments/3d_learned_attention/metrics.json")

ablation = pd.DataFrame([
    {"model": "1D-only (SELFIES Transformer)", "val_auroc": d1["auroc"]},
    {"model": "2D-only (GINEConv)", "val_auroc": d2["auroc"]},
    {"model": "3D-only (uniform mean, control)", "val_auroc": d3_uniform["auroc"]},
    {"model": "3D-only (Boltzmann weighted mean)", "val_auroc": d3_weighted["auroc"]},
    {"model": "3D-only (learned attention + Boltzmann prior)", "val_auroc": d3_attn["auroc"]},
    {"model": "Multimodal (concat baseline)", "val_auroc": auroc_concat},
    {"model": "Multimodal (gated fusion)", "val_auroc": auroc_gated},
]).sort_values("val_auroc", ascending=False).reset_index(drop=True)

ablation


,model,val_auroc
0,3D-only (learned attention + Boltzmann prior),0.886250
1,Multimodal (concat baseline),0.883750
2,2D-only (GINEConv),0.881250
3,"3D-only (uniform mean, control)",0.848125
4,1D-only (SELFIES Transformer),0.846250
5,Multimodal (gated fusion),0.846250
6,3D-only (Boltzmann weighted mean),0.825000
